# 대표 장르 추출 및 층화 작업

## 목적
Steam 인디 게임 9,692개 중 리뷰 감성 분석을 위한 샘플링 대상 게임을 선별하기 위해, 각 게임의 대표 장르를 추출한다.

## 배경
- Steam `appdetails` API의 `genres` 필드는 알파벳 순으로 정렬된 멀티레이블 리스트로, "주 장르" 개념이 없음
- 층화 샘플링을 위해 게임당 하나의 대표 장르 배정이 필요
- 대표 장르 선정 방식: **희귀 장르 우선** — 게임이 가진 장르 중 전체 데이터에서 등장 빈도가 가장 낮은 장르를 대표로 선정

## 한계
- 희귀 장르가 반드시 해당 게임의 핵심 장르를 의미하지는 않음
- **Adventure · Casual 버킷 대표성 주의**: 두 장르는 Steam에서 범용적으로 붙는 태그라 원본 게임 수 대비 대표 장르 배정 수가 크게 줄어든다 (Adventure 84% 감소, Casual 74% 감소). 해당 버킷은 '다른 장르가 없는 순수 Adventure/Casual'에 가까우며, Adventure/Casual 전체를 대표하지 않으므로 장르 간 비교 시 해석에 주의가 필요하다.
- Sports/Racing 버킷에는 해당 장르가 부수적인 게임이 포함될 수 있음
- Steam 데이터 특성상 완전한 대표 장르 선정은 불가능하며, 이 한계를 감안하여 해석 필요

## 라이브러리 임포트

In [46]:
import ast
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option('display.max_columns', None)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 데이터 로드 및 파생 컬럼 생성

`steam_indie_9692.csv`를 불러오고 분석에 필요한 파생 컬럼을 추가합니다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `positive_rate`: 긍정 리뷰 비율 (%)
- `genres`: 문자열 리스트 파싱

In [47]:
df = pd.read_csv('../../../data/preprocessed/steam_indie_games.csv')
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['positive_rate'] = df['positive'] / df['total_reviews'] * 100

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres'] = df['genres'].apply(parse_genres)

print(f'모집단: {len(df):,}개')
print(f'출시연도 분포:')
print(df['release_date'].dt.year.value_counts().sort_index().to_string())

모집단: 9,692개
출시연도 분포:
release_date
2023.0    3278
2024.0    4059
2025.0    1962


## 1. 전체 장르 현황 탐색

분석 대상 장르를 결정하기 위해 전체 장르의 게임 수 분포를 확인합니다.

In [48]:
genre_counts_raw = {}
for genres in df['genres']:
    for genre in genres:
        genre_counts_raw[genre] = genre_counts_raw.get(genre, 0) + 1

genre_df = pd.DataFrame(list(genre_counts_raw.items()), columns=['Genre', 'Count']).sort_values('Count', ascending=False)
display(genre_df)

,Genre,Count
2,Indie,144
1,Adventure,62
0,Action,56
4,Casual,51
5,Simulation,50
3,RPG,33
10,Strategy,28
9,Utilities,18
11,Massively Multiplayer,11
7,Design & Illustration,9


In [49]:
median_count = genre_df['Count'].median()
mean_count   = genre_df['Count'].mean()

print(f'--- 장르별 게임 수 통계 ---')
print(f'중앙값: {median_count:.1f}개')
print(f'평균:   {mean_count:.1f}개')
print(f'표준편차: {genre_df["Count"].std():.1f}개')
print(f'최소값: {genre_df["Count"].min()}개')
print(f'최대값: {genre_df["Count"].max()}개')

above_median = genre_df[genre_df['Count'] >= median_count]
below_median = genre_df[genre_df['Count'] <  median_count]

print(f'\n=== 중앙값 이상 장르 ({len(above_median)}개) ===')
print(above_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))
print(f'\n=== 중앙값 미만 장르 ({len(below_median)}개) ===')
print(below_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))

--- 장르별 게임 수 통계 ---
중앙값: 9.0개
평균:   25.2개
표준편차: 34.6개
최소값: 2개
최대값: 144개

=== 중앙값 이상 장르 (11개) ===
                Genre  Count
                Indie    144
            Adventure     62
               Action     56
               Casual     51
           Simulation     50
                  RPG     33
             Strategy     28
            Utilities     18
Massively Multiplayer     11
Design & Illustration      9
               Sports      9

=== 중앙값 미만 장르 (9개) ===
               Genre  Count
Animation & Modeling      7
           Education      5
   Software Training      4
    Game Development      4
    Audio Production      4
              Racing      3
    Video Production      3
       Photo Editing      2
      Web Publishing      2


## 2. 대상 장르별 게임 수 분포

8개 대상 장르(`TARGET_GENRES`) 각각에 몇 개의 게임이 속하는지 확인합니다.
한 게임이 여러 장르를 가질 수 있으므로 장르별 집계는 중복을 허용합니다.

In [50]:
TARGET_GENRES = {'Adventure', 'Casual', 'Action', 'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing'}

# 대상 장르만 필터링하여 게임 수 집계
target_genre_counts = {
    genre: sum(1 for genres in df['genres'] if genre in genres)
    for genre in TARGET_GENRES
}
target_genre_df = (
    pd.DataFrame(list(target_genre_counts.items()), columns=['genre', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

fig = px.bar(
    target_genre_df,
    x='genre',
    y='count',
    text='count',
    title='대상 장르별 게임 수 분포 (중복 허용)',
    labels={'genre': '장르', 'count': '게임 수'},
    color='count',
    color_continuous_scale='Blues',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_categoryorder='total descending',
    coloraxis_showscale=False,
    height=450,
)
fig.show()

print(target_genre_df.to_string(index=False))

     genre  count
 Adventure     62
    Action     56
    Casual     51
Simulation     50
       RPG     33
  Strategy     28
    Sports      9
    Racing      3


## 3. 대상 장르 선정 및 대표 장르 추출

분석 대상 장르는 게임 수가 100개 이상이며 실제 게임 장르에 해당하는 8개로 한정한다.

**제외 기준**

| 제외 장르 | 이유 |
|---|---|
| Indie | 거의 모든 게임에 포함되어 변별력 없음 |
| Massively Multiplayer | 게임 수 부족 |
| Design & Illustration, Animation & Modeling, Education, Software Training, Game Development, Video Production, Audio Production, Photo Editing, Web Publishing, Accounting | 장르별 게임 수 중앙값(21개) 미만으로 통계적 분석에 부적합 |
| Utilities | 도구 성 게임 제외 |
   
| 최종 대상 장르 |
|----------|
| Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing |

In [51]:
# 8개 대상 장르만 필터링
df['genres_filtered'] = df['genres'].apply(lambda g: [x for x in g if x in TARGET_GENRES])

# 대상 장르가 하나도 없는 게임 제외
df_filtered = df[df['genres_filtered'].map(len) > 0].copy()

# 8개 장르 내 희귀도 계산
genre_count = Counter(genre for genres in df_filtered['genres_filtered'] for genre in genres)
print('=== 장르별 게임 수 (희귀도 기준) ===')
for genre, count in sorted(genre_count.items(), key=lambda x: x[1]):
    print(f'  {genre:25s}: {count:,}')

# 희귀 장르 우선으로 대표 장르 선정
df_filtered['primary_genre'] = df_filtered['genres_filtered'].apply(
    lambda genres: min(genres, key=lambda g: genre_count[g])
)

print('\n=== 대표 장르 분포 ===')
print(df_filtered['primary_genre'].value_counts().to_string())

=== 장르별 게임 수 (희귀도 기준) ===
  Racing                   : 3
  Sports                   : 9
  Strategy                 : 28
  RPG                      : 33
  Simulation               : 50
  Casual                   : 51
  Action                   : 56
  Adventure                : 62

=== 대표 장르 분포 ===
primary_genre
Simulation    28
Action        26
Strategy      24
RPG           18
Casual        15
Adventure     12
Sports         7
Racing         3


## 4. 결과 확인

게임별 필터링된 장르 목록과 선정된 대표 장르를 확인한다.

In [52]:
df_filtered[['appid', 'name', 'genres_filtered', 'primary_genre']].head(20)

,appid,name,genres_filtered,primary_genre
26,402710,Osiris: New Dawn,"[Action, Adventure, RPG]",RPG
117,657480,Dark Skies: The Nemansk Incident,"[Action, Adventure]",Action
173,789150,Mask of Fury,"[Action, Adventure, Casual, Simulation, Sports]",Sports
282,1006710,Decent Icons 2,[Casual],Casual
300,1030760,SteamDolls - Prologue Demo FREE,[Action],Action
462,1163140,Shanghai Office Simulator,"[Adventure, Casual, RPG, Simulation, Strategy]",Strategy
506,1194250,Screaming Chicken: Ultimate Showdown,"[Action, Adventure, Casual, Simulation]",Simulation
562,1242700,墲人之境-无人之境,"[Action, Adventure, Casual, Racing, RPG, Simul...",Racing
639,1293130,Purry & Panther: Lost in Helsinki,"[Adventure, Casual, Simulation]",Simulation
752,1348760,KARM,[Action],Action


In [53]:
genre_dist = df_filtered['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']

fig = px.bar(
    genre_dist,
    x='genre',
    y='count',
    text='count',
    title='대표 장르별 게임 수 분포',
    labels={'genre': '장르', 'count': '게임 수'},
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_categoryorder='total descending')
fig.show()

## 5. 층화 추출

대표 장르(`primary_genre`) × 리뷰 신뢰도(`trust`) 두 축으로 층을 구성한다.

### 층화 변수 기준

| 축 | 층 | 기준 |
|---|---|---|
| 장르 | Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing | `primary_genre` |
| 리뷰 신뢰도 | high | `total_reviews` ≥ 381 (긍정률 오차 ±5% 이하) |
| 리뷰 신뢰도 | mid  | 39 ≤ `total_reviews` < 381 (±5~15%) |
| 리뷰 신뢰도 | low  | `total_reviews` < 39 (±15% 초과) |

리뷰 신뢰도 경계값은 Wilson Score 95% 신뢰 구간 최대 오차 기준으로 산출한다.

### 5-1. Wilson Score 경계값 계산

긍정률 95% CI 최대 오차(worst case: p=0.5)를 기준으로 두 경계값을 산출한다.

- `LOW_BOUNDARY` : 오차 ±15% 이하가 되는 최소 리뷰 수 → low/mid 경계
- `HIGH_BOUNDARY`: 오차 ±5% 이하가 되는 최소 리뷰 수 → mid/high 경계

In [54]:
Z = 1.96

def wilson_margin(n):
    p = 0.5
    denom = 1 + Z**2 / n
    return (Z / denom) * np.sqrt(p*(1-p)/n + Z**2/(4*n**2)) * 100

def find_n_for_margin(target_pct):
    for n in range(1, 10000):
        if wilson_margin(n) <= target_pct:
            return n
    return 10000

LOW_BOUNDARY  = find_n_for_margin(15)
HIGH_BOUNDARY = find_n_for_margin(5)

print(f'low  (±15% 초과) : total_reviews <  {LOW_BOUNDARY}개')
print(f'mid  (±5~15%)    : {LOW_BOUNDARY} ≤ total_reviews < {HIGH_BOUNDARY}개')
print(f'high (±5% 이하)  : total_reviews >= {HIGH_BOUNDARY}개')

low  (±15% 초과) : total_reviews <  39개
mid  (±5~15%)    : 39 ≤ total_reviews < 381개
high (±5% 이하)  : total_reviews >= 381개


### 5-2. 층 할당

`primary_genre`와 리뷰 신뢰도를 조합하여 각 게임에 층을 배정한다.

In [55]:
def assign_trust(n):
    if n >= HIGH_BOUNDARY:
        return 'high'
    elif n >= LOW_BOUNDARY:
        return 'mid'
    else:
        return 'low'

df_filtered['trust']   = df_filtered['total_reviews'].apply(assign_trust)
df_filtered['stratum'] = df_filtered['primary_genre'] + '_' + df_filtered['trust']

pop = df_filtered['stratum'].value_counts().sort_index()
N   = len(df_filtered)

print(f'모집단: {N:,}개\n')
print(f'{"층":<22} {"게임 수":>8}  {"비중":>7}')
print('-' * 42)
for stratum, cnt in pop.items():
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%')
print('-' * 42)
print(f'{"합계":<22} {N:>8,}  {100.0:>6.1f}%')

모집단: 133개

층                          게임 수       비중
------------------------------------------
Action_high                   2     1.5%
Action_low                   15    11.3%
Action_mid                    9     6.8%
Adventure_low                 9     6.8%
Adventure_mid                 3     2.3%
Casual_high                   1     0.8%
Casual_low                   12     9.0%
Casual_mid                    2     1.5%
RPG_high                      4     3.0%
RPG_low                       9     6.8%
RPG_mid                       5     3.8%
Racing_high                   1     0.8%
Racing_mid                    2     1.5%
Simulation_high               4     3.0%
Simulation_low               15    11.3%
Simulation_mid                9     6.8%
Sports_high                   2     1.5%
Sports_low                    3     2.3%
Sports_mid                    2     1.5%
Strategy_high                 5     3.8%
Strategy_low                 11     8.3%
Strategy_mid                  8     6.0%
---

### 5-3. 표본 배분 — 비례 배분 + 최소 하한선

순수 비례 배분은 게임 수가 많은 장르에 표본이 과도하게 집중된다.
이를 보완하기 위해 **층당 최소 하한선**을 적용하고, 총합이 목표 수를 초과하면 큰 층에서 비례 축소한다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `TOTAL_N` | 200 | 목표 표본 수 |
| `MIN_PER_STRATUM` | 5 | 층당 최소 추출 수 |
| 층 수 | 24 | 8장르 × 3신뢰도 |

In [56]:
TOTAL_N         = 200
MIN_PER_STRATUM = 5
RANDOM_SEED     = 42

def compute_sample_plan(pop_series, total_n, min_floor):
    proportional = (pop_series / pop_series.sum() * total_n).round().astype(int)
    floored = proportional.clip(lower=min_floor)
    floored = floored.combine(pop_series, min)
    overflow = floored.sum() - total_n
    if overflow > 0:
        reducible = floored[(floored > min_floor) & (floored < pop_series)]
        if len(reducible) > 0:
            above = reducible - min_floor
            cut   = (above / above.sum() * overflow).round().astype(int)
            diff  = cut.sum() - overflow
            if diff != 0:
                cut.iloc[cut.argmax()] -= diff
            floored[reducible.index] -= cut
    return floored

sample_plan  = compute_sample_plan(pop, TOTAL_N, MIN_PER_STRATUM)
proportional = (pop / pop.sum() * TOTAL_N).round().astype(int)

print(f'목표 표본: {TOTAL_N}개  |  층당 최소: {MIN_PER_STRATUM}개\n')
print(f'{"층":<22} {"모집단":>8}  {"비중":>7}  {"비례":>6}  {"최종":>6}  {"추출률":>7}  비고')
print('-' * 76)
for stratum in pop.index:
    cnt  = pop[stratum]
    prop = proportional[stratum]
    n    = sample_plan[stratum]
    rate = n / cnt * 100
    note = '전수' if n == cnt else ('하한' if n == MIN_PER_STRATUM and prop < MIN_PER_STRATUM else '')
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%  {prop:>6}  {n:>6}  {rate:>6.1f}%  {note}')
print('-' * 76)
print(f'{"합계":<22} {N:>8,}  {"100.0%":>7}  {proportional.sum():>6}  {sample_plan.sum():>6}')

SAMPLE_PLAN = sample_plan.to_dict()

목표 표본: 200개  |  층당 최소: 5개

층                           모집단       비중      비례      최종      추출률  비고
----------------------------------------------------------------------------
Action_high                   2     1.5%       3       2   100.0%  전수
Action_low                   15    11.3%      23      15   100.0%  전수
Action_mid                    9     6.8%      14       9   100.0%  전수
Adventure_low                 9     6.8%      14       9   100.0%  전수
Adventure_mid                 3     2.3%       5       3   100.0%  전수
Casual_high                   1     0.8%       2       1   100.0%  전수
Casual_low                   12     9.0%      18      12   100.0%  전수
Casual_mid                    2     1.5%       3       2   100.0%  전수
RPG_high                      4     3.0%       6       4   100.0%  전수
RPG_low                       9     6.8%      14       9   100.0%  전수
RPG_mid                       5     3.8%       8       5   100.0%  전수
Racing_high                   1     0.8%       2       1

### 5-4. 층화 추출

In [57]:
import pandas as pd

sampled_frames = []
for stratum, n in SAMPLE_PLAN.items():
    pool     = df_filtered[df_filtered['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print(f'추출 완료: {len(df_sample)}개')
print(df_sample['stratum'].value_counts().sort_index())

추출 완료: 133개
stratum
Action_high         2
Action_low         15
Action_mid          9
Adventure_low       9
Adventure_mid       3
Casual_high         1
Casual_low         12
Casual_mid          2
RPG_high            4
RPG_low             9
RPG_mid             5
Racing_high         1
Racing_mid          2
Simulation_high     4
Simulation_low     15
Simulation_mid      9
Sports_high         2
Sports_low          3
Sports_mid          2
Strategy_high       5
Strategy_low       11
Strategy_mid        8
Name: count, dtype: int64


### 5-5. 검증

층별 모집단 비중과 표본 비중을 비교하여 대표성을 확인한다.

In [58]:
pop_ratio  = df_filtered['stratum'].value_counts(normalize=True).sort_index() * 100
samp_ratio = df_sample['stratum'].value_counts(normalize=True).sort_index() * 100
samp_count = df_sample['stratum'].value_counts().sort_index()

print(f'=== 층별 모집단 vs 표본 비교 ===')
print(f'{"층":<22} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}  {"표본n":>6}')
print('-' * 58)
for stratum in pop_ratio.index:
    p   = pop_ratio[stratum]
    s   = samp_ratio.get(stratum, 0)
    n   = samp_count.get(stratum, 0)
    gap = s - p
    flag = ' ⚠' if abs(gap) > 10 else ''
    print(f'{stratum:<22} {p:>7.1f}%  {s:>6.1f}%  {gap:>+6.1f}%  {n:>6}{flag}')
print('-' * 58)

print(f'\n=== 체크리스트 ===')
checks = [
    ('층당 최소 하한 충족', all(samp_count >= MIN_PER_STRATUM)),
    ('총 표본 수',         len(df_sample) == TOTAL_N),
]
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')

=== 층별 모집단 vs 표본 비교 ===
층                          모집단%      표본%       격차     표본n
----------------------------------------------------------
Action_high                1.5%     1.5%    +0.0%       2
Action_low                11.3%    11.3%    +0.0%      15
Action_mid                 6.8%     6.8%    +0.0%       9
Adventure_low              6.8%     6.8%    +0.0%       9
Adventure_mid              2.3%     2.3%    +0.0%       3
Casual_high                0.8%     0.8%    +0.0%       1
Casual_low                 9.0%     9.0%    +0.0%      12
Casual_mid                 1.5%     1.5%    +0.0%       2
RPG_high                   3.0%     3.0%    +0.0%       4
RPG_low                    6.8%     6.8%    +0.0%       9
RPG_mid                    3.8%     3.8%    +0.0%       5
Racing_high                0.8%     0.8%    +0.0%       1
Racing_mid                 1.5%     1.5%    +0.0%       2
Simulation_high            3.0%     3.0%    +0.0%       4
Simulation_low            11.3%    11.3%    +0.

### 5-6. 표본 저장

층화 추출 결과를 CSV로 저장한다. 리뷰 수집 스크립트의 입력 파일로 사용된다.

In [59]:
OUT_COLS = [
    'appid', 'name', 'release_date', 'genres',
    'positive', 'negative', 'total_reviews', 'positive_rate', 'price',
    'developers', 'primary_genre', 'trust', 'stratum',
]

out_path = '../../../data/preprocessed/steam_indie_genre_stratified_sample.csv'
df_sample[OUT_COLS].to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_sample)}개)')
df_sample[OUT_COLS].head()

저장 완료 → ../../../data/preprocessed/steam_indie_genre_stratified_sample.csv (133개)


,appid,name,release_date,genres,positive,negative,total_reviews,positive_rate,price,developers,primary_genre,trust,stratum
0,3193840,OUTLAWED,NaT,"[Action, Indie]",660.0,379.0,1039.0,63.522618,14.99,Okami Studio,Action,high,Action_high
1,2524680,Akatori: Сhapter One,NaT,"[Action, Adventure, Indie]",884.0,63.0,947.0,93.347413,0.00,Code Wakers,Action,high,Action_high
2,2533930,(END) Made in Hell,NaT,"[Action, Indie]",8.0,5.0,13.0,61.538462,0.00,Team frog,Action,low,Action_low
3,2762290,Billy's Game Show,NaT,"[Action, Adventure, Indie]",27.0,0.0,27.0,100.000000,3.99,CH757,Action,low,Action_low
4,1348760,KARM,NaT,"[Action, Indie]",28.0,10.0,38.0,73.684211,1.99,Matthieu Gouby,Action,low,Action_low


## 6. 표본 장르 분포 확인

추출된 200개 표본의 장르 × 리뷰 신뢰도 분포를 시각화하여 층화 결과를 검증한다.

In [60]:
# ── 1. 장르별 표본 수 (bar chart) ──────────────────────────────────────────
genre_dist = df_sample['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']
pop_genre  = df_filtered['primary_genre'].value_counts().reset_index()
pop_genre.columns = ['genre', 'pop_count']
genre_dist = genre_dist.merge(pop_genre, on='genre')
genre_dist['sample_ratio'] = genre_dist['count'] / genre_dist['count'].sum() * 100
genre_dist['pop_ratio']    = genre_dist['pop_count'] / genre_dist['pop_count'].sum() * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=genre_dist['genre'], y=genre_dist['pop_ratio'],
    name='모집단 비중(%)', marker_color='lightsteelblue', opacity=0.7
))
fig.add_trace(go.Bar(
    x=genre_dist['genre'], y=genre_dist['sample_ratio'],
    name='표본 비중(%)', marker_color='steelblue',
    text=genre_dist['count'].apply(lambda x: f'{x}개'),
    textposition='outside'
))
fig.update_layout(
    title='장르별 모집단 vs 표본 비중 비교',
    xaxis_title='장르', yaxis_title='비중 (%)',
    barmode='group', height=450,
    xaxis_categoryorder='total descending'
)
fig.show()

In [61]:
# ── 2. 장르 × 신뢰도 히트맵 ────────────────────────────────────────────────
heatmap_data = (
    df_sample.groupby(['primary_genre', 'trust'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['high', 'mid', 'low'], fill_value=0)
)

fig2 = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=['high', 'mid', 'low'],
    y=heatmap_data.index.tolist(),
    text=heatmap_data.values,
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True,
))
fig2.update_layout(
    title='장르 × 리뷰 신뢰도 표본 수 히트맵',
    xaxis_title='리뷰 신뢰도',
    yaxis_title='장르',
    height=450
)
fig2.show()

In [62]:
# ── 3. 신뢰도별 표본 수 (pie chart) ────────────────────────────────────────
trust_dist = df_sample['trust'].value_counts().reindex(['high', 'mid', 'low'])

fig3 = go.Figure(go.Pie(
    labels=trust_dist.index,
    values=trust_dist.values,
    hole=0.4,
    marker_colors=['#2196F3', '#90CAF9', '#E3F2FD'],
    textinfo='label+percent+value'
))
fig3.update_layout(
    title='리뷰 신뢰도별 표본 비중',
    height=400
)
fig3.show()